# 🏆 Mora SP Cup 2026 — Team Zentrix: **v10 APEX — FULL POWER**
## 200-Epoch Fresh Start | All 460 Images | Progressive Loss Curriculum | CosineWarmRestarts × 4

### 🔥 Why v10 APEX Will Break Through 0.63+:
1. **🆕 Fresh Random Initialization**: v8/v9 were stuck in v7's local minimum. Fresh weights → completely new and potentially better minimum.
2. **📦 Full 460-Image Training (No Holdout)**: Competition test set is 20 *different* images we can't see. Holding 4 images out only wastes training signal. Train on ALL 460.
3. **📈 Progressive Loss Curriculum (3 Phases)**:
   - **Phase 1 (Ep 1–100): Pure Charbonnier** → Maximum PSNR optimization, no competing terms
   - **Phase 2 (Ep 101–160): Charbonnier + 0.05×MS-FFT** → Add high-frequency texture recovery
   - **Phase 3 (Ep 161–200): Charbonnier + 0.05×MS-FFT + 0.03×Soft-SSIM** → Lock structural fidelity
4. **🔄 CosineAnnealingWarmRestarts (T₀=50, T_mult=1)**: 4 full optimizer cycles — each reset escapes local minima others can't.
5. **⚡ EMA (β=0.999) + SWA (Final 20 Epochs)**: Dual averaging locks flat global minimum.
6. **⏱ ~160–180 min total** on Colab T4 GPU.

### 🚀 Quick Start:
1. **Runtime → Disconnect and delete runtime** (clean GPU session).
2. **Runtime → Run all** (`Ctrl + F9`).
3. `best_denoiser_v10.pth` will auto-download when done!

> **⚠️ NOTE:** No `best_denoiser_v7.pth` needed in Drive for v10 — completely fresh start!


In [ ]:
# [1] Verify GPU & Install
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    props = torch.cuda.get_device_properties(0)
    print(f'\u2705 GPU: {props.name} | VRAM: {props.total_memory/1024**3:.1f} GB')
else:
    raise RuntimeError('\u274c No GPU! Enable: Runtime → Change runtime type → T4 GPU.')
!nvidia-smi --query-gpu=name,memory.free,memory.total --format=csv
!pip install -q opencv-python-headless scikit-image PyWavelets

In [ ]:
# [2] Directories
!mkdir -p scripts/checkpoints models
!mkdir -p competition_data/public/noisy competition_data/public/ground_truth
print('\u2705 Directories ready!')

In [ ]:
# [3] NAFNet-Ultra v10 Architecture (Width=64, 86.49M Params) — scripts/models.py
%%writefile scripts/models.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class NAFBlock(nn.Module):
    def __init__(self, c, DW_Expand=2, FFN_Expand=2, drop_out_rate=0.0):
        super().__init__()
        dw = c * DW_Expand
        self.conv1 = nn.Conv2d(c, dw, 1, bias=True)
        self.conv2 = nn.Conv2d(dw, dw, 3, 1, 1, groups=dw, bias=True)
        self.conv3 = nn.Conv2d(dw//2, c, 1, bias=True)
        self.sca = nn.Sequential(nn.AdaptiveAvgPool2d(1),
                                  nn.Conv2d(dw//2, dw//2, 1, bias=True))
        self.sg = SimpleGate()
        ffn = FFN_Expand * c
        self.conv4 = nn.Conv2d(c, ffn, 1, bias=True)
        self.conv5 = nn.Conv2d(ffn//2, c, 1, bias=True)
        self.norm1 = nn.GroupNorm(1, c)
        self.norm2 = nn.GroupNorm(1, c)
        self.dropout1 = nn.Dropout(drop_out_rate) if drop_out_rate > 0 else nn.Identity()
        self.dropout2 = nn.Dropout(drop_out_rate) if drop_out_rate > 0 else nn.Identity()
        self.beta  = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))

    def forward(self, inp):
        x = self.norm1(inp)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.sg(x)
        x = x * self.sca(x)
        x = self.conv3(x)
        x = self.dropout1(x)
        y = inp + x * self.beta
        x = self.norm2(y)
        x = self.conv4(x)
        x = self.sg(x)
        x = self.conv5(x)
        x = self.dropout2(x)
        return y + x * self.gamma

class NAFNet(nn.Module):
    def __init__(self, img_channel=3, width=64,
                 enc_blks=(2, 2, 4, 8), dec_blks=(2, 2, 2, 2), middle_blk_num=8):
        super().__init__()
        self.intro  = nn.Conv2d(img_channel, width, 3, 1, 1, bias=True)
        self.ending = nn.Conv2d(width, img_channel, 3, 1, 1, bias=True)
        self.encoders, self.decoders = nn.ModuleList(), nn.ModuleList()
        self.ups, self.downs = nn.ModuleList(), nn.ModuleList()
        chan = width
        for num in enc_blks:
            self.encoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))
            self.downs.append(nn.Conv2d(chan, chan*2, 2, 2))
            chan *= 2
        self.middle_blks = nn.Sequential(*[NAFBlock(chan) for _ in range(middle_blk_num)])
        for num in dec_blks:
            self.ups.append(nn.Sequential(
                nn.Conv2d(chan, chan*2, 1, bias=False), nn.PixelShuffle(2)))
            chan //= 2
            self.decoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))
        self.padder_size = 2 ** len(enc_blks)

    def forward(self, inp):
        B, C, H, W = inp.shape
        ph = (self.padder_size - H % self.padder_size) % self.padder_size
        pw = (self.padder_size - W % self.padder_size) % self.padder_size
        x = F.pad(inp, (0, pw, 0, ph), 'reflect') if ph or pw else inp
        res = x
        x = self.intro(x)
        encs = []
        for enc, down in zip(self.encoders, self.downs):
            x = enc(x); encs.append(x); x = down(x)
        x = self.middle_blks(x)
        for dec, up, skip in zip(self.decoders, self.ups, reversed(encs)):
            x = up(x) + skip; x = dec(x)
        x = self.ending(x) + res
        return torch.clamp(x[:, :, :H, :W] if ph or pw else x, 0., 1.)

def build_denoising_model(width=64):
    m = NAFNet(img_channel=3, width=width,
               enc_blks=(2,2,4,8), dec_blks=(2,2,2,2), middle_blk_num=8)
    n = sum(p.numel() for p in m.parameters())
    print(f'[Model] NAFNet-Ultra v10 | Width={width} | Params={n/1e6:.2f}M')
    return m


In [ ]:
# [4] Full-Data Dataset with MixUp Augmentation — scripts/dataset.py
%%writefile scripts/dataset.py
import os, glob, random
import numpy as np, cv2, torch
from torch.utils.data import Dataset

class DenoisingPairDataset(Dataset):
    def __init__(self, noisy_dir, gt_dir, patch_size=256, augment=True, mixup_alpha=0.3):
        self.noisy_dir    = noisy_dir
        self.patch_size   = patch_size
        self.augment      = augment
        self.mixup_alpha  = mixup_alpha
        self.gt_files     = sorted(glob.glob(os.path.join(gt_dir, '*.png')))
        if not self.gt_files:
            raise ValueError(f'No GT images in {gt_dir}')
        print(f'[Dataset] Loaded {len(self.gt_files)} full training images (no holdout)')

    def __len__(self): return len(self.gt_files)

    def _load_pair(self, idx):
        gp  = self.gt_files[idx]
        bid = os.path.splitext(os.path.basename(gp))[0]
        np_ = os.path.join(self.noisy_dir, f'{bid}_noise.png')
        if not os.path.exists(np_):
            np_ = os.path.join(self.noisy_dir, f'{bid}.png')
        g = cv2.cvtColor(cv2.imread(gp),  cv2.COLOR_BGR2RGB)
        n = cv2.cvtColor(cv2.imread(np_), cv2.COLOR_BGR2RGB)
        return n, g

    def _crop_and_aug(self, n, g):
        h, w = n.shape[:2]
        p = self.patch_size
        y = random.randint(0, max(0, h-p))
        x = random.randint(0, max(0, w-p))
        n, g = n[y:y+p, x:x+p], g[y:y+p, x:x+p]
        if random.random() > 0.5: n, g = np.fliplr(n).copy(), np.fliplr(g).copy()
        if random.random() > 0.5: n, g = np.flipud(n).copy(), np.flipud(g).copy()
        k = random.randint(0, 3)
        if k: n, g = np.rot90(n, k).copy(), np.rot90(g, k).copy()
        return np.ascontiguousarray(n), np.ascontiguousarray(g)

    def __getitem__(self, idx):
        n, g = self._load_pair(idx)
        if self.augment:
            n, g = self._crop_and_aug(n, g)
            # MixUp augmentation
            if self.mixup_alpha > 0 and random.random() > 0.5:
                idx2 = random.randint(0, len(self.gt_files)-1)
                n2, g2 = self._load_pair(idx2)
                n2, g2 = self._crop_and_aug(n2, g2)
                lam = float(np.random.beta(self.mixup_alpha, self.mixup_alpha))
                n = (lam * n.astype(np.float32) + (1-lam) * n2.astype(np.float32)).astype(np.uint8)
                g = (lam * g.astype(np.float32) + (1-lam) * g2.astype(np.float32)).astype(np.uint8)
        return (torch.from_numpy(n).permute(2,0,1).float()/255.,
                torch.from_numpy(g).permute(2,0,1).float()/255.)


In [ ]:
# [5] Mount Google Drive & Load Full Dataset
from google.colab import drive
import os, glob
drive.mount('/content/drive')

matches = (glob.glob('/content/drive/MyDrive/**/competition_data/public', recursive=True) +
           glob.glob('/content/drive/MyDrive/**/public', recursive=True))
if matches:
    dp = matches[0]
    print(f'\u2705 Dataset found: {dp}')
    os.system(f'cp -rn {dp}/noisy/* competition_data/public/noisy/ 2>/dev/null')
    os.system(f'cp -rn {dp}/ground_truth/* competition_data/public/ground_truth/ 2>/dev/null')

nn_ = len(glob.glob('competition_data/public/noisy/*.png'))
ng  = len(glob.glob('competition_data/public/ground_truth/*.png'))
print(f'\U0001f4ca Dataset: {nn_} noisy | {ng} ground truth images')
if ng == 0: raise RuntimeError('No ground truth found! Check Drive path.')

In [ ]:
# [6] ===== v10 APEX: FULL POWER TRAINING ENGINE (200 EPOCHS) =====
# Fresh Init | ALL 460 Images | Progressive Loss | CosineWarmRestarts x4 | EMA + SWA
from scripts.models import build_denoising_model
from scripts.dataset import DenoisingPairDataset
import torch, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.swa_utils import AveragedModel, SWALR
import time, gc, os, copy, numpy as np

gc.collect(); torch.cuda.empty_cache()
device = torch.device('cuda')

# ─── HYPER-PARAMETERS ────────────────────────────────────────────
WIDTH        = 64       # 86.49M Parameters
PATCH_SIZE   = 256      # Fixed 256px — consistent training signal
BATCH_SIZE   = 4        # Batch per step
ACCUM_STEPS  = 4        # Effective batch = 16
TOTAL_EPOCHS = 200      # Full training
PEAK_LR      = 6e-4     # High LR allows escape from local minima in fresh training
MIN_LR       = 1e-7     # Cosine floor
EMA_DECAY    = 0.999
SWA_START_EP = 181      # SWA: final 20 epochs
SWA_LR       = 1e-5
# ─────────────────────────────────────────────────────────────────

# Progressive Loss Phase Boundaries
PHASE2_START = 101  # Add MS-FFT
PHASE3_START = 161  # Add Soft-SSIM

# Full dataset — NO validation holdout (competition test is separate)
full_ds = DenoisingPairDataset(
    'competition_data/public/noisy',
    'competition_data/public/ground_truth',
    patch_size=PATCH_SIZE, augment=True, mixup_alpha=0.3)
train_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)
print(f'\U0001f680 Full Training Power: {len(full_ds)} images | {len(train_loader)} batches/epoch')

# ─── Fresh Random Initialization ─────────────────────────────────
model = build_denoising_model(width=WIDTH).to(device)
print('[*] \U0001f195 FRESH RANDOM INITIALIZATION — Exploring new loss landscape!')
print('[*] Best-in-class benchmark to beat: Score=0.6205 | PSNR=30.8025 dB | SSIM=0.9179')

# ─── EMA & SWA ───────────────────────────────────────────────────
class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema_model = copy.deepcopy(model).eval()
        for p in self.ema_model.parameters(): p.requires_grad_(False)
        self.decay = decay
    def update(self, model):
        with torch.no_grad():
            for ep, mp in zip(self.ema_model.parameters(), model.parameters()):
                ep.data.mul_(self.decay).add_(mp.data, alpha=1.0 - self.decay)

ema = ModelEMA(model, decay=EMA_DECAY)
swa_model  = AveragedModel(model)
swa_active = False

# ─── Optimizer: CosineAnnealingWarmRestarts (T0=50, 4 cycles) ────
optimizer = optim.AdamW(model.parameters(), lr=PEAK_LR,
                         weight_decay=1e-4, betas=(0.9, 0.999))
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=50, T_mult=1, eta_min=MIN_LR)
swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR, anneal_epochs=5)
scaler = torch.amp.GradScaler('cuda')

# ─── Loss Functions ───────────────────────────────────────────────
def charbonnier(pred, gt, eps=1e-6):
    return torch.mean(torch.sqrt((pred - gt)**2 + eps))

def multiscale_fft_loss(pred, gt, scales=[1, 2, 4]):
    total = 0.0
    for s in scales:
        p_ = F.avg_pool2d(pred, s, s) if s > 1 else pred
        g_ = F.avg_pool2d(gt,   s, s) if s > 1 else gt
        total += torch.mean(torch.abs(
            torch.fft.rfft2(p_, norm='ortho') - torch.fft.rfft2(g_, norm='ortho')))
    return total / len(scales)

def soft_ssim_loss(pred, gt):
    C1, C2 = 0.01**2, 0.03**2
    mu_p = F.avg_pool2d(pred, 11, 1, 5)
    mu_g = F.avg_pool2d(gt,   11, 1, 5)
    mu_pp, mu_gg, mu_pg = mu_p**2, mu_g**2, mu_p*mu_g
    sg_pp = F.avg_pool2d(pred*pred, 11, 1, 5) - mu_pp
    sg_gg = F.avg_pool2d(gt*gt,     11, 1, 5) - mu_gg
    sg_pg = F.avg_pool2d(pred*gt,   11, 1, 5) - mu_pg
    return 1.0 - torch.mean(
        (2*mu_pg + C1) * (2*sg_pg + C2) / ((mu_pp + mu_gg + C1) * (sg_pp + sg_gg + C2)))

def get_loss(pred, gt, epoch):
    loss = charbonnier(pred, gt)
    if epoch >= PHASE2_START:
        loss = loss + 0.05 * multiscale_fft_loss(pred, gt)
    if epoch >= PHASE3_START:
        loss = loss + 0.03 * soft_ssim_loss(pred, gt)
    return loss

def get_phase(epoch):
    if epoch < PHASE2_START:   return 'Phase1: Charbonnier'
    elif epoch < PHASE3_START: return 'Phase2: +MS-FFT'
    else:                       return 'Phase3: +Soft-SSIM'

BASE_PSNR = 19.8257
BASE_SSIM = 0.4644

def psnr_metric(pred, gt):
    return 10. * torch.log10(1. / torch.mean((pred - gt)**2).clamp(1e-8)).item()

def ssim_metric(pred, gt):
    C1, C2 = 0.01**2, 0.03**2
    mu_p = F.avg_pool2d(pred, 11, 1, 5)
    mu_g = F.avg_pool2d(gt,   11, 1, 5)
    mu_pp, mu_gg, mu_pg = mu_p**2, mu_g**2, mu_p*mu_g
    sg_pp = F.avg_pool2d(pred*pred, 11, 1, 5) - mu_pp
    sg_gg = F.avg_pool2d(gt*gt,     11, 1, 5) - mu_gg
    sg_pg = F.avg_pool2d(pred*gt,   11, 1, 5) - mu_pg
    return torch.mean(
        (2*mu_pg + C1) * (2*sg_pg + C2) / ((mu_pp + mu_gg + C1) * (sg_pp + sg_gg + C2))).item()

def composite(psnr, ssim_v):
    return 0.6 * min(max((psnr - BASE_PSNR) / 15., 0.), 1.) + 0.4 * max(ssim_v - BASE_SSIM, 0.)

# Training-time quick eval (sample 8 random patches from batch)
def quick_eval(net, loader, n_batches=3):
    net.eval()
    p_list, s_list = [], []
    with torch.no_grad():
        for i, (nv, gv) in enumerate(loader):
            if i >= n_batches: break
            nv, gv = nv.to(device), gv.to(device)
            with torch.amp.autocast('cuda'):
                pv = net(nv)
            p_list.append(psnr_metric(pv, gv))
            s_list.append(ssim_metric(pv, gv))
    vp = sum(p_list) / len(p_list)
    vs = sum(s_list) / len(s_list)
    return vp, vs, composite(vp, vs)

CKPT_V10      = 'scripts/checkpoints/best_denoiser_v10.pth'
CKPT_DRIVE    = '/content/drive/MyDrive/best_denoiser_v10.pth'
CKPT_DRIVE_BK = '/content/drive/MyDrive/best_denoiser.pth'

best_score = 0.0  # Fresh start — any improvement is a win
best_psnr  = 0.0
best_ssim  = 0.0

print('=' * 90)
print(f'v10 APEX | Epochs: {TOTAL_EPOCHS} | Peak LR: {PEAK_LR} | WarmRestarts T0=50 (4 cycles)')
print(f'Phase 1: Ep 1–100 (Charbonnier) | Phase 2: Ep 101–160 (+FFT) | Phase 3: Ep 161–200 (+SSIM)')
print(f'SWA from Ep {SWA_START_EP} | EMA β={EMA_DECAY} | Full {len(full_ds)}-image training')
print('=' * 90)

# ─── MAIN 200-EPOCH TRAINING LOOP ────────────────────────────────
for epoch in range(1, TOTAL_EPOCHS + 1):
    # Phase banner at transitions
    if epoch in [PHASE2_START, PHASE3_START]:
        phase = get_phase(epoch)
        print(f'\n\U0001f4c8 LOSS CURRICULUM UPGRADE → {phase}')

    if epoch == SWA_START_EP:
        print(f'\n\U0001f300 SWA ACTIVATED at epoch {epoch}!')
        swa_active = True

    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    t0 = time.time()

    for i, (noisy, gt) in enumerate(train_loader):
        noisy, gt = noisy.to(device, non_blocking=True), gt.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            pred = model(noisy)
            loss = get_loss(pred, gt, epoch) / ACCUM_STEPS
        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS

        if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.01)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
            ema.update(model)

    if swa_active:
        swa_model.update_parameters(model)
        swa_scheduler.step()
    else:
        scheduler.step()

    cur_lr   = optimizer.param_groups[0]['lr']
    avg_loss = total_loss / len(train_loader)
    elapsed  = time.time() - t0
    model.eval()
    torch.cuda.empty_cache()

    # Evaluate both standard and EMA model on 3 random training batches
    val_p_std, val_s_std, comp_std = quick_eval(model, train_loader)
    val_p_ema, val_s_ema, comp_ema = quick_eval(ema.ema_model, train_loader)

    candidates = [
        (comp_std, val_p_std, val_s_std, model.state_dict(), '[STD]'),
        (comp_ema, val_p_ema, val_s_ema, ema.ema_model.state_dict(), '[EMA]'),
    ]
    if swa_active and epoch % 5 == 0:
        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)
        p_swa, s_swa, c_swa = quick_eval(swa_model, train_loader)
        candidates.append((c_swa, p_swa, s_swa, swa_model.state_dict(), '[SWA]'))

    cur_comp, cur_p, cur_s, best_w, tag = max(candidates, key=lambda x: x[0])
    flag = ''

    if cur_comp > best_score:
        best_score, best_psnr, best_ssim = cur_comp, cur_p, cur_s
        ckpt = {'model_state_dict': best_w, 'width': WIDTH,
                'psnr': cur_p, 'ssim': cur_s, 'composite_score': cur_comp,
                'epoch': epoch, 'type': tag, 'phase': get_phase(epoch)}
        torch.save(ckpt, CKPT_V10)
        torch.save(ckpt, CKPT_DRIVE)
        torch.save(ckpt, CKPT_DRIVE_BK)
        flag = f'  \U0001f3af RECORD! Score={cur_comp:.4f} | Saved to Drive'

    # Print every epoch, show cycle indicator
    cycle = ((epoch - 1) // 50) + 1
    print(f'Ep [{epoch:03d}/{TOTAL_EPOCHS}] C{cycle} {get_phase(epoch)[:6]} | '
          f'Loss: {avg_loss:.4f} | PSNR: {cur_p:.2f} dB | SSIM: {cur_s:.4f} | '
          f'Score: {cur_comp:.4f} {tag} | LR: {cur_lr:.1e} | {elapsed:.0f}s{flag}')

    # Cycle boundary banner
    if epoch % 50 == 0 and epoch < TOTAL_EPOCHS:
        print(f'\n\U0001f504 COSINE CYCLE {cycle} COMPLETE → RESTARTING LR to {PEAK_LR:.0e}!')
        print(f'   Best so far: Score={best_score:.4f} | PSNR={best_psnr:.2f} dB | SSIM={best_ssim:.4f}\n')
    torch.cuda.empty_cache()

# ─── Final SWA flush ─────────────────────────────────────────────
if swa_active:
    print('\n[*] Final SWA BatchNorm update...')
    torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)
    p_f, s_f, c_f = quick_eval(swa_model, train_loader)
    print(f'    SWA Final: PSNR={p_f:.4f} | SSIM={s_f:.4f} | Score={c_f:.4f}')
    if c_f > best_score:
        best_score, best_psnr, best_ssim = c_f, p_f, s_f
        ckpt = {'model_state_dict': swa_model.state_dict(), 'width': WIDTH,
                'psnr': p_f, 'ssim': s_f, 'composite_score': c_f, 'type': '[SWA-FINAL]'}
        torch.save(ckpt, CKPT_V10); torch.save(ckpt, CKPT_DRIVE); torch.save(ckpt, CKPT_DRIVE_BK)
        print('\U0001f300 SWA FINAL IS THE BEST MODEL!')

print('=' * 90)
print('\U0001f3c6 v10 APEX TRAINING COMPLETE!')
print(f'   Best Composite Score : {best_score:.4f}')
print(f'   Best PSNR            : {best_psnr:.4f} dB')
print(f'   Best SSIM            : {best_ssim:.4f}')
if best_score >= 0.6205:
    print(f'   \U0001f3c6 BEAT PREVIOUS CHAMPION v7! Gain: +{(best_score-0.6205):.4f}')
else:
    print(f'   Keep v7 as champion. Diff: {(best_score-0.6205):.4f}')
print('=' * 90)

In [ ]:
# [7] SHA-256 Checksum & Auto-Download
import hashlib
from google.colab import files

ckpt_path = CKPT_V10 if os.path.exists(CKPT_V10) else CKPT_DRIVE
sha = hashlib.sha256()
with open(ckpt_path, 'rb') as f:
    for chunk in iter(lambda: f.read(65536), b''): sha.update(chunk)
print('=' * 90)
print(f'SHA-256: {sha.hexdigest()}')
print(f'Best Score: {best_score:.4f} | PSNR: {best_psnr:.4f} dB | SSIM: {best_ssim:.4f}')
print('=' * 90)
print('\U0001f4e5 Downloading best_denoiser_v10.pth...')
files.download(ckpt_path)
print('\u2705 Done! Place file in your models/ folder.')